# Grounding Models in Feedback: Tools, Execution, and Principles
*How ReAct, RLEF, and Constitutional AI each build a self-improvement loop from a different kind of feedback — real-world actions, code execution, and AI-generated critique*

# Why Models Need to Act, Not Just Reason
Large language models are already very good at language tasks, and most people are familiar with using them as chatbots. But to be useful for real-world tasks, a model needs to be able to **interact with its environment** — tools, code, live data — and actually learn from what it finds there, not just reason from what it already knows.

This notebook covers three papers, each a different way for a model to improve itself, based on where the feedback comes from:
1. **ReAct** — feedback from interacting with a real environment (search, tools).
2. **RLEF** (grounding code LLMs in execution feedback) — feedback from actually running code and its tests.
3. **Constitutional AI** — feedback from the model critiquing its own output.


# The Problem: Reasoning and Acting Were Separate
When humans decide to do something, we usually think about it first, then act, then observe the result, and use that new information to think again. It's a loop: **think → act → observe → think again.**

Language models historically struggled to do this loop themselves. The core issue: a model reasoning purely from its own internal knowledge has no idea what's actually happening in the world right now, and can easily hallucinate. Ask a plain model "what's the weather today?" and it has no way to know — unless it can actually go search for the answer.

Before ReAct, there were two separate lines of work:
- **Chain-of-thought reasoning** — gives the model a way to show its steps, but those steps are based purely on what the model already "knows" internally, with no outside feedback.
- **Tool-using models** (like WebGPT) — learn to interact with things like web browsers, but without an explicit reasoning process guiding the actions.

**ReAct's idea:** combine both. Let the model reason *and* act, with each one feeding into the other.


# ReAct: Synergizing Reasoning and Acting

**Paper:** [arxiv.org/abs/2210.03629](https://arxiv.org/abs/2210.03629)

ReAct's method is simple: just prompt the model to alternate between **thinking** and **acting**.

1. Ask the model to think step by step about the task (a "thought").
2. Based on that thought, decide what action to take (e.g., search something).
3. Take the action and get an observation back from the environment.
4. Feed that observation into the next round of thinking.
5. Repeat, interleaved: thought 1 → act 1 → thought 2 → act 2 → ...

Everything happens in plain language — the "thoughts" are just text the model generates, and don't affect the environment directly. Only the actions do. This keeps the whole approach simple: no need to train a separate policy or reward model, just clever prompting.

## Why Interleaving (Not Just "Reason Then Act") Matters
Each step informs the next, similar to how a person naturally behaves: you're thirsty, so you walk to the kitchen; you find no water there, so you decide your next move based on that new information. It's sequential — each observation shapes the next thought, and each thought shapes the next action.


# How Do You Keep the Model's Actions Valid?
If a model can generate any action in plain text, how do you make sure it only picks from actions that actually make sense in your system?

One common approach: frame it as a **classification task** — give the model the current state and reasoning, along with a fixed list of valid actions, and have it choose from that list rather than generate free-form text. This keeps actions grounded and workable, and is used in other systems too (including some robotics work from the same research group), where only certain physical actions are actually possible.


# Worked Example: Why Interleaving Beats Reasoning-Only or Acting-Only
The question: *"Aside from the Apple Remote, what other device can control the program Apple Remote was originally designed to interact with?"* (from the HotpotQA dataset)

- **Standard prompting** (just ask the question): wrong answer.
- **Chain-of-thought** (ask it to think step by step): still doesn't reliably get it right — the model has to rely only on what it already "knows," which may be incomplete or wrong.
- **Actions only** (search calls without reasoning in between): the model searches "Apple Remote," then searches "Front Row" (a term it saw in the result), but ends up following the wrong thread and never lands on a good answer.
- **ReAct (interleaved thought + action):**
  1. *Thought:* "I need to search Apple Remote, and find what program it was designed to interact with."
  2. *Action:* Search "Apple Remote" → learns it was designed to control the Front Row media center.
  3. *Thought:* "Now I need to search Front Row."
  4. *Action:* Search "Front Row" → doesn't find it directly, so tries "Front Row software" instead.
  5. *Thought:* Realizes Front Row is discontinued media software, and is able to answer correctly.

The reasoning step at each stage helps the model figure out *what* to search next and *how to interpret* what it finds — rather than either guessing blindly (chain-of-thought alone) or searching without any real plan (actions alone).

**Where you can see this today:** if you turn on "thinking mode" in models like Qwen, you'll see this exact interleaved thinking-and-tool-calling behavior happen automatically — because these models have been trained (distilled) on traces that look like this.


# Open Questions About ReAct

## Does the Model Know What It Already Knows?
If a model has already seen a fact during training (like a Wikipedia page about "Front Row"), why would it still go search for it instead of just answering directly? This connects to a bigger, unresolved question: **do models know what they know?** Opinions differ — some believe models are quite confident about their own knowledge, but a common finding is that models tend to be **overconfident** and poorly calibrated when asked to rate their own certainty. This is still an open research problem. For practical purposes, the more useful question isn't "does the model know what it knows" — it's "can we get the model to reliably use the right tools" to stay grounded, regardless of that uncertainty.

## What Happens With Noisy or Contradictory Search Results?
ReAct doesn't guarantee the environment's feedback is correct or consistent. If search results disagree with each other, that's ultimately something the system (or the person building it) has to handle — for example, by using majority voting across multiple attempts, or adding validation steps before trusting an answer. The goal for a real application isn't full certainty about the model's internal state — it's building enough guardrails that the final answer is reliable. Hallucination grounded in outside search results is generally easier to catch and control than hallucination coming purely from a model's internal state.

## Is the "Action Space" Part of the Model's Reasoning, or Separate?
ReAct treats reasoning ("thoughts") and acting as two separate loops rather than one merged action space — thoughts live in language space and don't affect the environment; actions are what actually changes the environment (like a search call). Since language models are trained on language, it tends to help to have reasoning expressed as tokens in this same space, which then guides the model toward picking the right action — rather than trying to represent "thinking" as some other kind of internal, non-language representation.


# Results: Knowledge Tasks (HotpotQA and FEVER)
These are the benchmarks tested:
- **HotpotQA:** multi-hop question answering over Wikipedia (answering a question that requires connecting facts across multiple pages).
- **FEVER:** fact-checking — deciding whether a claim is true, false, or unverifiable.

Both tasks have a simple, constrained action space: essentially "search a Wikipedia page," "look up a specific string," or "finish."

## Methods Compared
- **Standard prompting:** no thoughts, no actions, no observations.
- **Chain-of-thought (CoT):** think step by step, but no outside actions.
- **CoT + self-consistency (CoT-SC):** majority voting across multiple CoT attempts.
- **Act-only:** take actions, but no explicit reasoning steps in between.
- **ReAct:** the full interleaved thought-and-action approach — with variants that can fall back to CoT-SC if ReAct doesn't succeed within a certain number of steps, or vice versa (fall back to ReAct if CoT-SC's majority answer isn't confident enough).

## Key Findings
- ReAct consistently beat **act-only** prompting.
- ReAct did **not always** beat plain chain-of-thought — it outperformed CoT on FEVER, but not consistently on HotpotQA.
- **Combining** ReAct with CoT-SC (using either as a fallback for the other) beat both individual approaches.
- **Chain-of-thought's main failure mode was hallucination** — since it has no way to check itself against the real world. ReAct's grounding in actual search results made it noticeably more trustworthy, since it could verify information rather than guessing.
- **Fine-tuning ReAct (rather than just prompting it) improved results further** — and adding an outer training loop on top does even better still.

**Takeaway:** there's real value in combining a model's internal knowledge with outside, retrieved knowledge, using reasoning as the glue that decides what to retrieve and how to interpret it.


# Results: Decision-Making Tasks (WebShop)
WebShop is a simulated online shopping environment: the model is given an instruction (e.g., "find a nightstand with drawers") and has to take a sequence of actions — searching, clicking, comparing options — to actually complete the purchase.

## Baselines Compared
- **Imitation learning (IL):** essentially supervised fine-tuning on human demonstrations.
- **IL + RL:** imitation learning followed by reinforcement learning on top.
- **ReAct**

## Results
ReAct outperformed both IL and IL+RL on both **score** (partial credit for making progress through intermediate steps) and **success rate** (fully completing the task). However, it still fell well short of expert humans — in one comparison, ReAct scored 66.6 versus 82.1 for human experts, showing there's real headroom left. Since this is a multi-step task, a mistake at any single step can cascade and hurt the final success rate more than the intermediate score.

*(Note: these specific ReAct-vs-human numbers may come from a particular fine-tuned configuration in the paper rather than the baseline few-shot prompting setup — worth checking the original paper's tables directly if you need the precise figure for a specific configuration.)*


# Trade-offs of ReAct
- **Large action spaces are a problem:** if there are too many possible actions, you need more few-shot demonstrations than can reasonably fit in the model's context window.
- **Higher inference cost:** since ReAct requires multiple reasoning-and-acting steps rather than one shot, it costs more to run than simpler prompting.
- **But overall:** it clearly improves performance on question answering, fact-checking, and decision-making tasks, while reducing hallucination and producing more interpretable step-by-step traces than reasoning alone.


# Extending ReAct: Other Ideas

## What If the Environment Gives Noisy or Misleading Feedback?
Possible fixes: add a **reflection** step, where the agent explicitly checks whether the feedback it just got actually makes sense, rather than blindly trusting it. Also important: the ability to **backtrack** — if a chain of thought and action isn't leading anywhere useful (which noisy feedback can cause), the agent needs a way to abandon that path rather than loop on it repeatedly. Another option: build better confidence signals by repeating a task multiple times and taking the most consistent (highest-frequency) result, rather than trusting a single noisy pass.

## What Other Human Thinking Patterns Could Inform Agent Design?
Humans don't reason the same way for every task — sometimes we lean on past experience, sometimes we reason step by step from scratch, and it depends heavily on the situation. A few directions worth exploring beyond plain ReAct:
- **Task decomposition:** break a complex task into smaller subtasks before applying reasoning-and-acting to each piece.
- **Parallel thinking paths:** explore more than one line of reasoning at once, rather than committing to a single sequential thread.
- **Memory of past experience:** let the agent draw on prior interactions, not just the current episode.
- **Tuning the reasoning-to-action ratio per task:** some tasks may benefit from a lot of upfront thinking before acting; others may not need much at all. It's an open question whether this ratio could be learned automatically per task, rather than fixed. This connects to a real, current problem: some modern reasoning models "overthink" even simple tasks, producing unnecessarily long reasoning traces — an active area of research is figuring out how to prevent that.
- **Matching tasks to model strengths:** different models may be better at different sub-skills (tool calling, chain-of-thought reasoning, delegating work), so a system could benchmark models on subtasks and route work to whichever model is strongest at that particular piece — essentially building a compound system rather than relying on one model for everything.


# RLEF: Grounding Code LLMs in Execution Feedback

**Paper:** [arxiv.org/abs/2410.02089](https://arxiv.org/abs/2410.02089)

When building coding agents, one of the most useful kinds of feedback for a reinforcement learning loop is **execution feedback** — actually running the code and seeing whether it works. RLEF was one of the first papers to show that this kind of feedback leads to real, large improvements in coding LLMs.

## Why This Matters
A large share of everyday engineering and coding work is increasingly being handed off to coding agents. For these agents to be genuinely useful, they need to understand what the user actually wants, and they need a way to check and improve their own code after generating it — not just generate one attempt and stop.

## The Basic Setup
This is an end-to-end reinforcement learning framework for code generation:
- **Actions:** the code the model generates.
- **Observations:** feedback from actually running that code against tests.
- **Reward:** binary — did the tests pass or fail?

Based on this feedback, the model is fine-tuned (using PPO, a reinforcement learning method) to get better over time.


# How the Feedback Loop Works
1. The model is given a plain-language problem description (e.g., "write a program that does X").
2. It generates a code solution.
3. That solution is tested against a small **public** set of tests.
4. If it fails, the failure feedback is given back to the model, and it tries again — this repeats until the code passes, or the model runs out of allowed attempts (turns).
5. Once a solution passes the public tests (or the turn limit is hit), it's tested against a separate, **private** set of tests — and *that* result determines the actual training reward used in the PPO/RL update.

This creates two phases: an "exploitation" phase, where the model repeatedly tries to solve the problem using its current abilities and the public-test feedback, and an "update" phase, where the model's underlying policy actually gets improved based on the private-test result.

## Worked Example: Detecting Palindrome Substrings
1. **Turn 1:** the model writes a basic solution — but it fails the public tests due to a timeout (a very common real-world coding issue).
2. The failure feedback is given back to the model.
3. **Turn 2:** the model writes an optimized version that fixes the timeout issue and passes the public tests.
4. This improved solution is then submitted to the hidden private tests, which determine the actual training reward.


# Why Split Tests Into "Public" and "Private"?
This separation is one of the paper's key design choices, and it matters for a specific reason: it stops the model from being trained directly on the same tests it's using to iterate and improve during generation. The public tests give fast, immediate feedback during the generation process (a small subset, kept small so iteration stays fast); the private tests stay completely hidden during generation and are only used afterward, to actually score the model.

This means the model can't simply memorize test outputs — it has to genuinely use the execution feedback to get better, rather than gaming the visible tests.


# A Technical Detail: Token-Level Actions, Turn-Level Rewards
Since this is a language model, it generates code one token at a time — giving fine-grained control over what gets generated. But the *reward* isn't computed per token. Instead, it's computed once per full turn (i.e., once per complete response), using the score from the last token of that response — meaning every token in that turn shares the same "advantage" value in the RL update, rather than getting individually scored feedback.

(For anyone familiar with other RL training methods: this token-vs-turn distinction is similar to the difference between per-token reward assignment and sequence-level reward assignment used in methods like GSPO.)


# Does It Actually Work? Results
Measuring **solve rate** (the fraction of problems solved, out of a batch of sampled attempts — e.g., "solve rate 10 at k" means passing at least one out of a batch, tracked against how large that batch is) against sampling budget: models trained with RLEF on CodeContests (a competitive programming dataset) clearly solved more problems than models without RLEF training — and this held on a log scale, meaning the improvement wasn't just a small bump, it compounded as the sampling budget grew.

## Why Does It Help?
Plain base models generally don't benefit much from just seeing faulty solutions and execution feedback on their own. What actually seems to help is training the model on CodeContests **while explicitly showing it the execution feedback at every turn** — and this benefit **generalizes** to other coding benchmarks beyond CodeContests, not just the one it was trained on.

This becomes clear when comparing how many errors happen in turn 1, turn 2, and turn 3 of the iteration process: with RLEF, later turns clearly have fewer wrong outputs than earlier ones — the model is genuinely learning to repair its own mistakes across turns, not just generating a fresh, unrelated guess each time. Without this iterative loop, the edits made in later turns aren't nearly as targeted or successful.


# Open Questions About RLEF

## Is Binary Pass/Fail Feedback Detailed Enough?
Since the feedback is just "pass" or "fail" (not, say, a full error trace or a suggested fix), does the model have enough signal to actually improve? For the relatively simple problems used in this paper (each solution under about 100 lines of code), binary feedback seems to be enough. For harder, more complex problems, richer feedback — like the actual error trace or other debugging metadata — would likely be needed to make repair efficient. This is flagged as a promising area for further research.

## Does the Model Actually Get Better at Solving Problems on the First Try?
Since the reward is binary and only based on the final result, does this approach actually push the model to solve problems correctly in one shot, or does it just get better at iterative repair? It likely depends on how hard the problem is — harder problems may still take multiple turns even after training. But since the model does get several chances to self-correct during the exploitation phase (using the inference-time public-test feedback) before its final answer is scored, it has real opportunity to improve within an episode, not just across training runs. The number of turns allowed is itself a tunable setting that could be adjusted for harder problems.

## Process vs. Outcome Rewards, Again
Should feedback be given at every single step, rather than just once at the end? This connects to the earlier process-reward-model vs. outcome-reward-model comparison covered in verification research — that debate isn't fully settled, and the right choice likely depends on the specific domain and benchmark.

## Why Does RLEF Generalize to Other Benchmarks?
One possibility: the model is learning from feedback that's harder and more varied than what it might see in typical human-level training data, which could be why the resulting skill transfers well. A related question: could supervised fine-tuning alone (without RL) get similar gains? Fine-tuning on reasoning/error-correction traces can likely capture some of this benefit for problems similar to what it was trained on, but RL-based training tends to generalize a bit further to genuinely newer problem types, since the loss function used in supervised fine-tuning inherently ties performance closer to the training distribution.


# The Bigger Picture
The core takeaway: there's a real, workable way to bring execution feedback into training and make code generation better through it — and it's not just a narrow trick. Trained on competitive programming problems specifically, this approach generalizes well to other code-generation benchmarks too, not just the one it was trained on.


# Beyond RLEF: Handling Large Codebases
A related, practical challenge: what happens when an entire codebase doesn't fit into a model's context window? A few ideas for handling this:

- **Search and iterate:** rather than trying to load everything at once, the model can use search tools to look for what's actually relevant, keep iterating until it decides it has enough information, and only then move on to generating a fix.
- **Summarize before combining:** summarize each relevant piece of code first, so that when combined, the summaries fit within the context window even if the raw code wouldn't.
- **Similarity-based retrieval:** build a structured representation of the codebase (e.g., a graph), and use similarity search to pull in only the most relevant pieces, iterating and verifying whether enough information has been gathered before generating an answer.

This is close to how real coding agent tools already work in practice — searching for relevant code, applying a patch, and then checking whether it actually passes tests. **SWE-bench** is one benchmark that specifically targets this kind of large-codebase, patch-and-verify workflow, and related work like **CodeMonkeys** ([arxiv.org/abs/2501.14723](https://arxiv.org/abs/2501.14723)) explores similar ideas.


# Constitutional AI: Learning from AI Feedback

**Paper:** [arxiv.org/abs/2212.08073](https://arxiv.org/abs/2212.08073)

The goal here is different from the first two papers: instead of grounding a model in an external tool or environment, this is about improving the model's **harmlessness** — its ability to avoid generating harmful outputs — while staying **helpful**.

## The Scaling Problem With Human Feedback
The standard way to improve a chatbot is RLHF: show humans two model outputs, ask which is better (more correct, more useful, more specific), and use those preferences to train a reward model that guides further training. This clearly works — as model size increases, RLHF-trained models give noticeably better responses than plain fine-tuned ones.

The problem: this doesn't scale well. Collecting tens of thousands of human preference labels is slow and expensive.

## The Idea: Replace Some Human Feedback With AI Feedback
Since models have gotten good at following instructions and reasoning about their own outputs, it becomes possible to have the model **critique and revise its own responses** based on a written set of principles — called a **Constitution** — rather than needing a human to judge every single output. Humans are only needed once, to write the Constitution itself, not to label every output afterward.

This works because models have become reliable at two specific skills: following formatting/behavioral instructions, and truthfully answering questions about their own output (e.g., "does this response contain X?").


# The Constitution and the Critique-Revise Loop
The paper defines **16 principles** (the "Constitution") describing desired model behavior. These aren't just abstract rules — they're used as concrete prompts that ask the model to check its own output and then fix it.

## Example Critique-and-Revision Prompts
- *Critique:* "Does this response contain anything harmful or unethical?" → *Revision:* "Remove anything harmful or unethical."
- *Critique:* "Does this response show any gender bias?" (with reasoning about why) → *Revision:* "Remove any trace of that bias."
- *Critique:* "Is this response inappropriate for young children?" → *Revision:* "Rewrite it to be appropriate."

Each of these requires the model to do two things: **identify** whether a certain behavior is present, and then **follow an instruction** to fix it in a specific way. Humans only set the principles up front — the actual critique-and-revision loop runs on the model itself.


# Two Training Stages

## Stage 1: Supervised Learning (Self-Critique and Revision)
The model generates an output, critiques itself against the Constitution, revises the output, and is then fine-tuned on these self-corrected responses. Just from a number of these revision rounds, harmlessness improves — though helpfulness tends to decline somewhat as you push further in this direction, since strictly enforcing fixed principles naturally makes some responses less flexible. The overall combination of helpfulness and harmlessness still improves.

## Stage 2: Reinforcement Learning from AI Feedback (RLAIF)
This stage replaces RLHF's human-labeled preference data with **AI-generated** preference data:
1. Sample pairs of responses from the fine-tuned model.
2. Use a model (guided by the Constitution) to judge which response is better.
3. Train a **preference model** on these AI judgments.
4. Fine-tune the LLM using RL, with the preference model as the reward signal.

This whole process is called **RLAIF** (Reinforcement Learning from AI Feedback) — the same basic structure as RLHF, but with the human preference-labeling step replaced by a Constitution-guided AI judge.


# Do Harmlessness and Helpfulness Actually Trade Off?
Measuring **helpfulness Elo** (how much humans prefer a response for helpfulness) against **harmlessness Elo** (how much humans prefer it for being harmless), across several training approaches:
- **Helpfulness-only RLHF:** pushes helpfulness scores up.
- **Helpfulness + harmlessness RLHF (human feedback for both):** pushes harmlessness up too, at some cost to helpfulness.
- **Constitutional AI (supervised stage only):** still worse than full RLHF.
- **Constitutional RL with chain-of-thought reasoning added to the critique step:** this reaches the best overall trade-off (Pareto frontier) between helpfulness and harmlessness — outperforming the purely human-feedback-based approaches.

So: replacing human preference labels with AI feedback (guided by a written Constitution) not only avoided the scaling problem of human labeling, it actually reached a *better* combined outcome than standard RLHF, especially once chain-of-thought reasoning was added to the critique step. Harmlessness in particular improved substantially — a technique that has since become a strong contributor to the Claude models' safety behavior specifically, and has been adopted more broadly since.

**One nuance:** chain-of-thought reasoning in the critique step was associated with somewhat lower helpfulness scores than some other variants — likely reflecting the same general trade-off (pushing harder on harmlessness costs a bit of helpfulness), rather than chain-of-thought itself being a problem.


# Open Questions About Constitutional AI

## What Happens When the Constitution Needs to Change?
If principles need updating over time, how do you do that efficiently — and can you be sure the old rules are actually removed, not just weakened? Post-training (which includes this kind of Constitution training) is typically a small fraction of total training compute compared to pre-training, so it can happen fairly frequently as models are updated. But precisely controlling what a model "forgets" or fully stops following is a genuinely open research problem — related work on interpretability explores whether specific knowledge can be selectively suppressed, but it's not proven that a model can be made to fully forget something it learned.

## Is the Self-Critique Step in Stage 1 Really "Feedback"?
In the supervised stage, the model critiques and revises its own output, and is then fine-tuned directly on those self-corrected traces — there's no external reward or human check at that specific step; the model is just reinforcing its own generated attempt at following the principles. As long as this fine-tuning stays small relative to the model's original pre-training, it shifts the distribution of what the model tends to output without erasing its broader capabilities.

## How Do You Know the AI Feedback Is Actually Accurate?
Since human labels are being replaced with AI judgments, how do you verify the preference model trained on that AI feedback is any good? In practice, some human validation is still used — checking that the preference model's judgments are reasonably consistent with human judgments on a held-out set — even though the bulk of the training feedback comes from AI rather than humans.


# Related Follow-On Work
A few directions that built on Constitutional AI's core idea (using AI-generated feedback for self-improvement):
- Direct comparisons of **RLAIF vs. RLHF** as general training approaches.
- **Self-Refine** — iterative refinement of a model's own output using self-feedback.
- Work on teaching models to **self-correct through reinforcement learning**, more broadly than just the helpfulness/harmlessness framing here.

One useful observation from this line of work: getting a single model to critique itself can be harder than it sounds, since models can be overconfident and not fully aware of their own blind spots. Using a **consensus of multiple models** to critique a given output can work better than relying on just one model's self-judgment.


# Pulling the Three Papers Together
Each paper in this notebook is a different way of building a feedback loop that helps a model improve beyond what it learned purely from its original training data:

- **ReAct:** grounds the model by letting it act in the real world (search, tools) and reason about what it finds — useful for question answering, fact-checking, and decision-making, and it produces highly interpretable, step-by-step traces.
- **RLEF:** grounds code generation in **execution feedback** — actually running the code and its tests — letting the model iteratively repair its own mistakes, which reduces how many samples are needed to reach strong performance on tasks like competitive programming.
- **Constitutional AI:** grounds model behavior in a written set of principles, replacing large-scale human labeling with AI-generated feedback guided by those principles — reaching a better helpfulness/harmlessness trade-off than standard human-feedback-only RLHF.

The common thread: once a feedback loop has *enough real signal* — whether from an external tool, executable code, or a set of principles the model can reason about — it becomes possible to keep improving a model well beyond the boundaries of its original training data.


# A Few Broader Open Questions
- **How much should agent design borrow from human cognitive science?** There's real value in patterns like task decomposition, breaking problems into steps, and reasoning in parallel — largely because these mirror how humans solve problems. But a deeper open question is whether the entire process of *searching* through reasoning space can eventually be automated end-to-end, rather than hand-designed. This tends to be easier in domains with a well-defined action space (like games) than in open-ended domains involving real tools and real-world observations.
- **Will explicit frameworks like ReAct become unnecessary as RL-based post-training keeps improving?** Possibly, but not entirely — if a task's space of possible actions can be well-defined, more automated approaches can take over. But for domain-specific work (e.g., a finance agent, a legal agent), what the "right steps" even look like is often not well-defined, and something like ReAct's explicit workflow (reason → act → reason → act) still provides a useful, human-understandable structure to build on.
- **Can these three self-improvement techniques transfer to full agents (not just single model outputs)?** They're all fundamentally about generating tokens and supervising them with some reward signal, so at an abstract level, yes — the same ideas apply. But an agent typically also needs things like persistent memory and multi-session context, which go beyond what any of these three papers directly address on their own.
- **Could harmfulness be reduced through filtering or restrictions on training data, rather than post-training critique/revision?** This is already done to some extent, but since these models are trained on extremely large amounts of internet data, no amount of filtering fully removes every possible source of harmful content — meaning post-training techniques like Constitutional AI remain necessary as a complementary layer, not a replacement for careful data curation.
